# Deep Learning for Agricultural Crop Classification

---

## Objective
Build a Deep Learning model to classify agricultural crop images using CNN and Transfer Learning techniques.

---

## Problem Statement
Given an image of a crop, the model should correctly identify the crop category.

---

## Type of Problem
- Supervised Learning
- Multi-Class Image Classification

---

## Workflow
1. Dataset Loading & Understanding  
2. Image Preprocessing  
3. Model Building (CNN + Transfer Learning)  
4. Model Training  
5. Model Evaluation  
6. Error Analysis  
7. Inference System  

---

## Dataset Source
Kaggle Dataset: Agricultural Crops Image Classification

## Import Required Libraries

In [ ]:
# Install Kaggle API (only once)
!pip install -q kaggle

# Core Libraries
import os
import numpy as np
import matplotlib.pyplot as plt

# Deep Learning
import tensorflow as tf

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

print("Libraries Loaded Successfully ✅")

## Kaggle Authentication

To download the dataset, upload your `kaggle.json` file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data_dir = "/content/drive/MyDrive/Agri_DS/Agricultural-crops"

## Verify Dataset Structure

Each folder represents one class.

In [ ]:
import os

classes = os.listdir(data_dir)

print("Classes found:", classes)
print("Total classes:", len(classes))

In [ ]:
for cls in classes:
    class_path = os.path.join(data_dir, cls)
    print(f"{cls} → {len(os.listdir(class_path))} images")

## Image Preprocessing & Data Preparation

In this step, we prepare images for training by:

- Resizing images to a fixed size
- Normalizing pixel values (0–1)
- Applying data augmentation techniques
- Splitting dataset into training and validation sets

---

### Why is this important?

- Deep Learning models require fixed-size inputs  
- Normalization improves convergence  
- Data augmentation helps prevent overfitting  

## Define Image Parameters

In [ ]:
IMG_SIZE = 224   # Standard size for pretrained models
BATCH_SIZE = 32

## Data Augmentation & Normalization

We use ImageDataGenerator to:

- Normalize pixel values
- Apply augmentation (rotation, flipping, zoom)
- Automatically split data into training and validation sets

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,              # Normalize pixels
    validation_split=0.2,        # 80% train, 20% validation

    # Data Augmentation
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

## Load Training Data

In [ ]:
train_data = train_datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

##  Load Validation Data

In [ ]:
val_data = train_datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

## Class Index Mapping

This shows how class labels are encoded into numbers.

In [ ]:
print(train_data.class_indices)

## Verify Data Shape

We check:
- Image tensor shape
- Label shape

In [ ]:
images, labels = next(train_data)

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

## Visualizing Augmented Images

This helps verify that augmentation is working correctly.

In [ ]:
plt.figure(figsize=(10,10))

for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images[i])
    plt.axis("off")

plt.show()

## Key Observations

- Images are successfully resized to 224x224
- Pixel values are normalized between 0 and 1
- Data augmentation introduces variations such as:
  - Rotation
  - Flipping
  - Brightness changes

---

## Why Augmentation Matters

- Prevents overfitting
- Improves generalization
- Simulates real-world variations

---

## Conclusion of Step 2

- Dataset is now ready for training
- Images are preprocessed and augmented
- Training and validation sets are prepared

## Custom CNN Model

In this step, we build a Convolutional Neural Network (CNN) from scratch.

---

### Why CNN?

CNNs are specifically designed for image data because they:
- Capture spatial features (edges, textures, shapes)
- Reduce parameters using shared weights
- Automatically learn feature hierarchies

---

### CNN Architecture Overview

1. Convolution Layers → Feature extraction  
2. Activation (ReLU) → Non-linearity  
3. Pooling Layers → Downsampling  
4. Flatten Layer → Convert to vector  
5. Dense Layers → Classification  
6. Softmax → Output probabilities  

## Building the CNN Model

In [ ]:
from tensorflow.keras import layers, models

cnn_model = models.Sequential([

    # 🔹 First Convolution Block
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D((2,2)),

    # 🔹 Second Convolution Block
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    # 🔹 Third Convolution Block
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    # 🔹 Flattening
    layers.Flatten(),

    # 🔹 Fully Connected Layer
    layers.Dense(128, activation='relu'),

    # 🔹 Output Layer
    layers.Dense(train_data.num_classes, activation='softmax')
])

## Compile the Model

We define:
- Optimizer
- Loss function
- Evaluation metric

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Model Training

In this step, we train the CNN model using the prepared dataset.

---

### Goals:
- Learn patterns from training data
- Monitor performance on validation data
- Avoid overfitting

---

### Metrics to Track:
- Training Accuracy
- Validation Accuracy
- Training Loss
- Validation Loss

## Train CNN Model

In [ ]:
EPOCHS = 10

history_cnn = cnn_model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)

## Training Performance Visualization

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history):
    plt.figure(figsize=(12,5))

    # Accuracy Plot
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title("Accuracy vs Epochs")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()

    # Loss Plot
    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title("Loss vs Epochs")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()

    plt.show()

plot_history(history_cnn)

## Transfer Learning

In this step, we use a pretrained deep learning model instead of training from scratch.

---

### What is Transfer Learning?

Transfer Learning uses models trained on large datasets (like ImageNet) and adapts them to new tasks.

---

### Why Use It?

- Faster training  
- Better accuracy  
- Works well with smaller datasets  

---

### Model Used:
MobileNetV2 (lightweight and efficient)

## Load Pretrained MobileNetV2

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,     # remove original classifier
    weights='imagenet'
)

## Freeze Base Model Layers

We prevent pretrained weights from updating during training.

In [ ]:
base_model.trainable = False

## Add Custom Classification Layers

In [ ]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

transfer_model = models.Model(inputs=base_model.input, outputs=output)

## Model Summary

In [ ]:
transfer_model.summary()

## Compile Transfer Model

In [ ]:
transfer_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Train Transfer Learning Model

In [ ]:
history_transfer = transfer_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

## Performance Visualization

In [ ]:
plot_history(history_transfer)

## Why Transfer Learning Works Better

- Pretrained on millions of images (ImageNet)
- Already learned:
  - Edges
  - Shapes
  - Textures


##  Advantage Over CNN

| Feature | Custom CNN | Transfer Learning |
|--------|-----------|------------------|
| Training Time | High | Low |
| Accuracy | Medium | High |
| Data Requirement | High | Low |



## Limitation

- May not fully adapt to domain-specific features

## Model Evaluation

In this step, we evaluate the performance of the trained model.



### Goals:
- Measure how well the model performs
- Identify strengths and weaknesses
- Analyze class-wise performance



### Metrics Used:
- Accuracy
- Confusion Matrix
- Precision, Recall, F1 Score

## Generate Predictions

In [ ]:
# Reset validation generator
val_data.reset()

# Predict
predictions = transfer_model.predict(val_data)

# Convert probabilities → class index
y_pred = np.argmax(predictions, axis=1)

# True labels
y_true = val_data.classes

## Classification Report

Includes:
- Precision
- Recall
- F1 Score

In [ ]:
from sklearn.metrics import classification_report

class_labels = list(val_data.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_labels))

## Confusion Matrix

Shows correct and incorrect predictions across classes.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## Overall Accuracy

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true, y_pred)
print("Model Accuracy:", accuracy)

## Interpreting Results

### High Accuracy Means:
- Model is correctly classifying most images


### Confusion Matrix Insights:
- Diagonal → Correct predictions
- Off-diagonal → Misclassifications



### Precision:
- Out of predicted class, how many are correct?

### Recall:
- Out of actual class, how many detected?

### F1 Score:
- Balance between precision and recall


## Common Issues:

- Confusion between visually similar crops
- Low performance on underrepresented classes


## Conclusion

- Model performance evaluated successfully
- Identified strengths and weaknesses
- Ready for error analysis

## Error Analysis & Inference System

In this step, we:
- Analyze model errors
- Understand misclassifications
- Build a system to predict new images



### Why This Matters?

- Helps improve model performance
- Shows real understanding of model behavior
- Makes the project usable in real-world scenarios

## Visualizing Misclassified Images

In [ ]:
# Collect all validation images and labels
all_images = []
all_labels = []

for i in range(len(val_data)):
    imgs, labels = val_data[i]
    all_images.append(imgs)
    all_labels.append(labels)

# Convert to numpy arrays
all_images = np.vstack(all_images)
all_labels = np.vstack(all_labels)

# Convert labels from one-hot → index
all_labels = np.argmax(all_labels, axis=1)

In [ ]:
misclassified_indices = np.where(y_pred != all_labels)[0]

print("Total Misclassified:", len(misclassified_indices))

In [ ]:
plt.figure(figsize=(12,10))

for i, idx in enumerate(misclassified_indices[:9]):
    plt.subplot(3,3,i+1)

    plt.imshow(all_images[idx])

    true_label = class_labels[all_labels[idx]]
    pred_label = class_labels[y_pred[idx]]

    plt.title(f"True: {true_label}\nPred: {pred_label}")
    plt.axis("off")

plt.show()

## Gradio Interface for Crop Prediction

This interface allows users to upload an image and get:
- Predicted crop type
- Confidence score

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_with_confidence(img):
    img = img.resize((224,224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = transfer_model.predict(img_array)[0]

    class_idx = np.argmax(prediction)
    confidence = np.max(prediction)

    return {
        class_labels[i]: float(prediction[i]) for i in range(len(class_labels))
    }

interface = gr.Interface(
    fn=predict_with_confidence,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=3),
    title="🌾 Crop Classification System",
    description="Upload a crop image to classify it using Deep Learning"
)

interface.launch()